# HydroTwin Linear Regression Baseline

This notebook develops Linear Regression baseline models for two hydropower guide bearing components:

1. Turbine Guide Bearing (TGB)
2. Upper Guide Bearing (UGB)

Each sample uses the previous 36 observations of four input variables, representing six hours of operating history.

The prediction target is the component vibration at the following 10 minute interval.

Linear Regression is used as the baseline model against which LSTM and XGBoost will later be compared.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

## 2. Project Paths

The prepared TGB and UGB datasets from the data preparation stage are loaded for baseline model development.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PREPARED_DIR = (
    PROJECT_ROOT /
    "data" /
    "prepared"
)

TGB_DIR = (
    PREPARED_DIR /
    "tgb"
)

UGB_DIR = (
    PREPARED_DIR /
    "ugb"
)

MODEL_DIR = (
    PROJECT_ROOT /
    "models" /
    "linear_regression"
)

RESULTS_DIR = (
    PROJECT_ROOT /
    "results"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nLinear Regression models:")
print(MODEL_DIR)

## 3. Load Prepared TGB Data

The unscaled TGB training and validation sequences are loaded. November test data is not used during model development.

In [ ]:
X_train_tgb = np.load(
    TGB_DIR / "X_train.npy"
)

y_train_tgb = np.load(
    TGB_DIR / "y_train.npy"
)

X_val_tgb = np.load(
    TGB_DIR / "X_val.npy"
)

y_val_tgb = np.load(
    TGB_DIR / "y_val.npy"
)

train_times_tgb = np.load(
    TGB_DIR / "train_times.npy",
    allow_pickle=True
)

val_times_tgb = np.load(
    TGB_DIR / "val_times.npy",
    allow_pickle=True
)

In [ ]:
print("TGB")

print("X_train:", X_train_tgb.shape)
print("y_train:", y_train_tgb.shape)

print("X_val:", X_val_tgb.shape)
print("y_val:", y_val_tgb.shape)

In [ ]:
X_train_ugb = np.load(
    UGB_DIR / "X_train.npy"
)

y_train_ugb = np.load(
    UGB_DIR / "y_train.npy"
)

X_val_ugb = np.load(
    UGB_DIR / "X_val.npy"
)

y_val_ugb = np.load(
    UGB_DIR / "y_val.npy"
)

train_times_ugb = np.load(
    UGB_DIR / "train_times.npy",
    allow_pickle=True
)

val_times_ugb = np.load(
    UGB_DIR / "val_times.npy",
    allow_pickle=True
)

In [ ]:
print("UGB")

print("X_train:", X_train_ugb.shape)
print("y_train:", y_train_ugb.shape)

print("X_val:", X_val_ugb.shape)
print("y_val:", y_val_ugb.shape)

## 4. Convert Sequential Inputs to Tabular Inputs

Linear Regression requires two dimensional tabular input. Therefore, each 36 by 4 historical sequence is flattened into 144 predictors.

The same historical information is retained. Only the input representation is changed.

In [ ]:
X_train_tgb_flat = X_train_tgb.reshape(
    X_train_tgb.shape[0],
    -1
)

X_val_tgb_flat = X_val_tgb.reshape(
    X_val_tgb.shape[0],
    -1
)

X_train_ugb_flat = X_train_ugb.reshape(
    X_train_ugb.shape[0],
    -1
)

X_val_ugb_flat = X_val_ugb.reshape(
    X_val_ugb.shape[0],
    -1
)

In [ ]:
print("TGB training:")
print(
    X_train_tgb.shape,
    "->",
    X_train_tgb_flat.shape
)

print("\nUGB training:")
print(
    X_train_ugb.shape,
    "->",
    X_train_ugb_flat.shape
)

In [ ]:
print(
    "TGB train finite:",
    np.isfinite(
        X_train_tgb_flat
    ).all()
)

print(
    "TGB validation finite:",
    np.isfinite(
        X_val_tgb_flat
    ).all()
)

print(
    "UGB train finite:",
    np.isfinite(
        X_train_ugb_flat
    ).all()
)

print(
    "UGB validation finite:",
    np.isfinite(
        X_val_ugb_flat
    ).all()
)

## 5. TGB Linear Regression Model

A Linear Regression model is trained using the June to August TGB training sequences.

September remains separate and is used for validation.

In [ ]:
tgb_lr_model = LinearRegression()

In [ ]:
tgb_lr_model.fit(
    X_train_tgb_flat,
    y_train_tgb
)

print(
    "TGB Linear Regression training complete."
)

In [ ]:
tgb_val_pred = (
    tgb_lr_model.predict(
        X_val_tgb_flat
    )
)

In [ ]:
print(
    "Predictions:",
    tgb_val_pred.shape
)

print(
    "Actual:",
    y_val_tgb.shape
)

## 6. TGB Validation Performance

The baseline model is evaluated on the September validation dataset using RMSE, MAE, MAPE, and R squared.

In [ ]:
tgb_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_tgb,
        tgb_val_pred
    )
)

tgb_val_mae = mean_absolute_error(
    y_val_tgb,
    tgb_val_pred
)

tgb_val_mape = (
    mean_absolute_percentage_error(
        y_val_tgb,
        tgb_val_pred
    )
    * 100
)

tgb_val_r2 = r2_score(
    y_val_tgb,
    tgb_val_pred
)

In [ ]:
print(
    f"TGB Linear Regression Validation RMSE: "
    f"{tgb_val_rmse:.4f}"
)

print(
    f"TGB Linear Regression Validation MAE: "
    f"{tgb_val_mae:.4f}"
)

print(
    f"TGB Linear Regression Validation MAPE: "
    f"{tgb_val_mape:.2f}%"
)

print(
    f"TGB Linear Regression Validation R²: "
    f"{tgb_val_r2:.4f}"
)

In [ ]:
plot_points = 500

plt.figure(
    figsize=(14, 5)
)

plt.plot(
    y_val_tgb[-plot_points:],
    label="Actual TGB Vibration"
)

plt.plot(
    tgb_val_pred[-plot_points:],
    label="Linear Regression Prediction"
)

plt.xlabel(
    "Validation Observation"
)

plt.ylabel(
    "TGB Vibration"
)

plt.title(
    "TGB Linear Regression Validation Prediction"
)

plt.legend()

plt.tight_layout()

plt.show()

## 7. UGB Linear Regression Model

A separate Linear Regression baseline is trained for the Upper Guide Bearing using the same modelling procedure.

In [ ]:
ugb_lr_model = LinearRegression()

In [ ]:
ugb_lr_model.fit(
    X_train_ugb_flat,
    y_train_ugb
)

print(
    "UGB Linear Regression training complete."
)

In [ ]:
ugb_val_pred = (
    ugb_lr_model.predict(
        X_val_ugb_flat
    )
)

In [ ]:
print(
    "Predictions:",
    ugb_val_pred.shape
)

print(
    "Actual:",
    y_val_ugb.shape
)

## 8. UGB Validation Performance

The UGB baseline is evaluated using the same metrics and validation period as the TGB baseline.

In [ ]:
ugb_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_ugb,
        ugb_val_pred
    )
)

ugb_val_mae = mean_absolute_error(
    y_val_ugb,
    ugb_val_pred
)

ugb_val_mape = (
    mean_absolute_percentage_error(
        y_val_ugb,
        ugb_val_pred
    )
    * 100
)

ugb_val_r2 = r2_score(
    y_val_ugb,
    ugb_val_pred
)

In [ ]:
print(
    f"UGB Linear Regression Validation RMSE: "
    f"{ugb_val_rmse:.4f}"
)

print(
    f"UGB Linear Regression Validation MAE: "
    f"{ugb_val_mae:.4f}"
)

print(
    f"UGB Linear Regression Validation MAPE: "
    f"{ugb_val_mape:.2f}%"
)

print(
    f"UGB Linear Regression Validation R²: "
    f"{ugb_val_r2:.4f}"
)

In [ ]:
plt.figure(
    figsize=(14, 5)
)

plt.plot(
    y_val_ugb[-plot_points:],
    label="Actual UGB Vibration"
)

plt.plot(
    ugb_val_pred[-plot_points:],
    label="Linear Regression Prediction"
)

plt.xlabel(
    "Validation Observation"
)

plt.ylabel(
    "UGB Vibration"
)

plt.title(
    "UGB Linear Regression Validation Prediction"
)

plt.legend()

plt.tight_layout()

plt.show()

## 9. Linear Regression Baseline Summary

The validation performance of the TGB and UGB baseline models is summarized before final test evaluation.

In [ ]:
lr_validation_results = pd.DataFrame({
    "Component": [
        "TGB",
        "UGB"
    ],

    "Model": [
        "Linear Regression",
        "Linear Regression"
    ],

    "RMSE": [
        tgb_val_rmse,
        ugb_val_rmse
    ],

    "MAE": [
        tgb_val_mae,
        ugb_val_mae
    ],

    "MAPE": [
        tgb_val_mape,
        ugb_val_mape
    ],

    "R2": [
        tgb_val_r2,
        ugb_val_r2
    ]
})

lr_validation_results

In [ ]:
joblib.dump(
    tgb_lr_model,
    MODEL_DIR /
    "tgb_linear_regression.pkl"
)

joblib.dump(
    ugb_lr_model,
    MODEL_DIR /
    "ugb_linear_regression.pkl"
)

In [ ]:
print(
    "TGB model saved:",
    (
        MODEL_DIR /
        "tgb_linear_regression.pkl"
    ).exists()
)

print(
    "UGB model saved:",
    (
        MODEL_DIR /
        "ugb_linear_regression.pkl"
    ).exists()
)

In [ ]:
lr_validation_results.to_csv(
    RESULTS_DIR /
    "linear_regression_validation_results.csv",
    index=False
)

print(
    "Validation results saved successfully."
)